# Tutorial 
This notebook shows how to run PCR Simulation

In [49]:
import numpy as np
import pandas as pd
import plotly.express as px

from datetime import datetime, timedelta
from scripts import erosion, helper, shoreline, slr, storm

In [196]:
import importlib
importlib.reload(storm)
importlib.reload(helper)
importlib.reload(slr)
importlib.reload(shoreline)

<module 'scripts.shoreline' from 'c:\\Users\\pba003\\Documents\\00projects\\pcr_python\\scripts\\shoreline.py'>

In [64]:
# set up a seed 
np.random.seed(42)

## All in one go
Import historical wave time series. This data can be acquired from wave hindcast or Buoy data

In [126]:
# test out the function using data from data/wave_srilanka.csv
# import wave data
wave_data = pd.read_csv('data/wave_srilanka.csv')

# get hs, dir, tp, time from dataframe
time = wave_data.iloc[:, 0].values
hs = wave_data.iloc[:, 1].values
dir = wave_data.iloc[:, 2].values
tp = wave_data.iloc[:, 3].values
ts_hs = 95
ts_dur = 12.0

# detect storm 
storms, storms_ts = storm.detect(hs, dir, tp, time, ts_hs, ts_dur)

# fit storm and gap 
fitted_storms = storm.fit_storm(storms)
fitted_gap = storm.fit_gap_monsoon(storms)

# generate storm sample
storms_sample = storm.generate(
    fitted_storm=fitted_storms, 
    sampling_size=1000, 
    oversample=0.1, 
    max_dur=np.max(storms.duration))

# add gaps
storms_sample = storm.sampling_gap_ecdf(
    fitted_gap=fitted_gap, 
    storms_sample=storms_sample
)

# simulating one sample 
date_start = datetime(
    year=2000, 
    month=1, 
    day=1
)

date_end = datetime(
    year=2100,
    month=12, 
    day=31
)

# generate storm time series from date start to date end
synthetic_storm = storm.generate_monsoon_ts(
    date_start=date_start, 
    date_end=date_end, 
    storms_sample=storms_sample, 
    fitted_gap=fitted_gap
)

# simulate sea level rise 
synthetic_storm['slr'] = slr.simulate_slr(
    synthetic_storm=synthetic_storm, 
    date_start=date_start, 
    scenario='RCP85', 
    wl0=0
)

# calculate storm-induced erosion
_, synthetic_storm['erosion_storm'] = erosion.mendoza(synthetic_storm)

# calculate recovery 
synthetic_storm['recovery'] = shoreline.calculate_recovery(
    storms=synthetic_storm,
    rec_rate=7/365
)

# calculate retreat due to slr 
synthetic_storm['slr_retreat'] = shoreline.calculate_slr_retreat(
    storms=synthetic_storm, 
    m=0.024
)

# track shoreline evolution 
shoreline_track = shoreline.track_shoreline(synthetic_storm)

In [96]:
shoreline_track['time'] = helper.date_add_days(date_start, shoreline_track['day'])

px.line(
    shoreline_track,
    x='time', 
    y='shoreline_position', 
    labels={'time': 'Time', 'shoreline_position': 'Shoreline position (m)'},
    title=f'Shoreline Position'
)

## Monte carlo simulations

In [281]:
# monte carlo simulations
date_start = datetime(
    year=2000, 
    month=1, 
    day=1
)

date_end = datetime(
    year=2100,
    month=12, 
    day=31
)

nr_simulation = 1000
nr_batch = 100

# storm char
yearly_storm = 8
nr_storm = (date_end.year - date_start.year) * yearly_storm

# initialize an array 
shoreline_stats = np.empty((101, nr_simulation))
# shoreline_stats = []

sim_count = 0
max_dur = np.max(storms.duration)

while sim_count < nr_simulation:
    sampling_size = nr_storm * nr_batch

    # generate storm sample
    storms_sample = storm.generate(
        fitted_storm=fitted_storms, 
        sampling_size=sampling_size, 
        oversample=0.1, 
        max_dur=max_dur)

    # add gaps
    storms_sample = storm.sampling_gap_ecdf(
        fitted_gap=fitted_gap, 
        storms_sample=storms_sample
    )

    storm_count = 0

    for sim in range(nr_batch):

        # generate storm time series from date start to date end
        synthetic_storm = storm.generate_monsoon_ts(
            date_start=date_start, 
            date_end=date_end, 
            storms_sample=storms_sample, 
            fitted_gap=fitted_gap, 
            start_storm=storm_count
        )

        storm_count += len(synthetic_storm)

        # simulate sea level rise 
        synthetic_storm['slr'] = slr.simulate_slr(
            synthetic_storm=synthetic_storm, 
            date_start=date_start, 
            scenario='RCP85', 
            wl0=0
        )

        # calculate storm-induced erosion
        _, synthetic_storm['erosion_storm'] = erosion.mendoza(synthetic_storm)

        # calculate recovery 
        synthetic_storm['recovery'] = shoreline.calculate_recovery(
            storms=synthetic_storm,
            rec_rate=10/365
        )

        # calculate retreat due to slr 
        synthetic_storm['slr_retreat'] = shoreline.calculate_slr_retreat(
            storms=synthetic_storm, 
            m=0.024
        )

        # track shoreline evolution 
        shoreline_track = shoreline.track_shoreline(synthetic_storm)

        row = shoreline.get_annual_statistics(shoreline_track, kind='mean', date_start=date_start)
        
        try:
            shoreline_stats[:, sim_count] = row.flatten()
            print('a')
        except: 
            sim_count -= 1 # if there are any missing year, re-do the simulation
            print('b')

        # if sim_count == 0:
        #     shoreline_stats.append(row)
        # elif shoreline_stats[sim_count-1].shape[0] == row.shape[0]:
        #     shoreline_stats.append(row)
        # else: 
        #     sim_count -= 1

        sim_count += 1

        if sim_count >= nr_simulation:
            break


a
a
b
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
b
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a
a


In [280]:
shoreline_stats[:, sim_count] = row.flatten()

In [279]:
shoreline_stats[:, 1].shape

(101,)

In [ ]:
shoreline_stats[1, 2]

(1000,)